In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT = "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project"

## Understanding the Existing EMG Pipeline

I will now inspect the repository to understand the EMG pipeline. My goal is to identify the flow from raw EMG data to prediction, including preprocessing, windowing, model specifics, input/output shapes, dependencies, and execution instructions. I will use the provided `PROJECT_CONTEXT.md` file for additional context.

First, I will list the contents of the `00_Reference/Original_GitHub_Repo/` directory to get an overview of the project structure.

In [9]:
import os

repo_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/')

print(f"Checking repository path: {repo_path}")

if not os.path.exists(repo_path):
    print(f"Error: The repository path '{repo_path}' does not exist. Please check the path and try again.")
elif not os.path.isdir(repo_path):
    print(f"Error: The path '{repo_path}' is not a directory.")
else:
    contents = os.listdir(repo_path)
    if not contents:
        print(f"The directory '{repo_path}' exists but is empty.")
    else:
        print(f"Contents of the repository: {repo_path}")
        for root, dirs, files in os.walk(repo_path):
            level = root.replace(repo_path, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 4 * (level + 1)
            for f in files:
                print(f'{subindent}{f}')

Checking repository path: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/00_Reference/Original_Github_Repo/
Contents of the repository: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/00_Reference/Original_Github_Repo/
/
    README.md
src/
    preprocessing.py
    data_preparation.py
data/
    subject_1/
        S1_R2_G2.mat
        S1_R6_G4.mat
        S1_R5_G4.mat
        S1_R8_G3.mat
        S1_R8_G4.mat
        S1_R7_G1.mat
        S1_R7_G4.mat
        S1_R3_G1.mat
        S1_R10_G2.mat
        S1_R5_G2.mat
        S1_R9_G3.mat
        S1_R9_G1.mat
        S1_R6_G1.mat
        S1_R6_G3.mat
        S1_R3_G4.mat
        S1_R8_G1.mat
        S1_R10_G4.mat
        S1_R1_G2.mat
        S1_R3_G3.mat
        S1_R7_G3.mat
        S1_R3_G2.mat
        S1_R4_G3.mat
        S1_R6_G2.mat
        S1_R8_G2.mat
        S1_R1_G1.mat
        S1_R2_G4.mat
        S1_R4_G2.mat
        S1_R9_G4.mat
        S1_R7_G2.mat
        S1_R4_G4.mat
        S1_R5_G1.

## Inspecting the `README.md` file

To understand the overall project, its dependencies, and how it is expected to be executed, I will now read the `README.md` file from the `Original_Github_Repo/`.

In [10]:
import os

readme_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/README.md')

if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        readme_content = f.read()
    print(readme_content)
else:
    print(f"Error: README.md not found at '{readme_path}'")

# Light-Weight Vision Language Model Guided Gesture Recognition Based on Electromyography

### VLM for Gesture Classification

## Overview

<p align="center">
    <img src="images/1.png" style="max-width:500px">
</p>


>**Abstract**: <br>
> Active rehabilitation of the upper limbs requires the decoding of user intentions from biosensors, particularly electromyography (EMG), to deliver augmented support during activities of daily living (ADL). Traditional methods typically depend on supervised, subject-specific training to reduce accuracy degradation caused by anatomical and environmental variations. However, these approaches often neglect the context in which gestures are intended. To bridge this gap, we propose a novel gesture decoding pipeline that integrates vision understanding to provide contextual information, thereby enhancing the control of objects during gesture inference. Our decoding pipeline is optimized for edge deployment and is evaluated for its ability to infer gestures

## Deep Dive into Preprocessing (`src/preprocessing.py`)

To understand the initial stages of the EMG pipeline, I will now inspect the content of `src/preprocessing.py`. This will help us determine how the raw `.mat` data is processed before being fed into the models, and clarify the expected input format for subsequent steps.

In [11]:
import os

preprocessing_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/src/preprocessing.py')

if os.path.exists(preprocessing_path):
    with open(preprocessing_path, 'r') as f:
        preprocessing_content = f.read()
    print(preprocessing_content)
else:
    print(f"Error: preprocessing.py not found at '{preprocessing_path}'")

import numpy as np
from scipy.signal import butter, iirnotch, filtfilt

class EMGPreprocessing:
    def __init__(self, fs=200, notch_freq=60.0, low_cut=5.0, high_cut=99.0, quality_factor=30.0, order=4):
        self.fs = fs
        self.notch_freq = notch_freq
        self.low_cut = low_cut
        self.high_cut = high_cut
        self.quality_factor = quality_factor
        self.order = order

    def remove_mains(self, data: np.ndarray) -> np.ndarray:
        """
        Apply a notch filter to remove the main frequency interference from the data.

        Parameters:
        data (ndarray): The input data array to be filtered. Shape (n_samples, n_channels).

        Returns:
        ndarray: The filtered data with the main frequency interference removed.
        """
        b, a = iirnotch(self.notch_freq, self.quality_factor, self.fs)
        return filtfilt(b, a, data, axis=1)

    def bandpass_filter(self, data: np.ndarray) -> np.ndarray:
        """
        Apply a bandpass filt

## Deep Dive into Data Preparation (`src/data_preparation.py`)

Now that we understand the preprocessing steps, I will inspect the content of `src/data_preparation.py`. This file is expected to handle the windowing of the preprocessed EMG data and potentially extract features, which are crucial steps before feeding the data into the machine learning model.

In [12]:
import os

data_preparation_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/src/data_preparation.py')

if os.path.exists(data_preparation_path):
    with open(data_preparation_path, 'r') as f:
        data_preparation_content = f.read()
    print(data_preparation_content)
else:
    print(f"Error: data_preparation.py not found at '{data_preparation_path}'")

import numpy as np
import os
from scipy.io import loadmat
from typing import List, Dict, Tuple
from torch.utils.data import Dataset

class EMGDataPreparation:
    """
    A class used to prepare EMG data for machine learning tasks.

    Attributes
    ----------
    base_path : str
        The base directory path where the EMG data is stored.
    fs : int
        The sampling frequency of the EMG data.
    rec_time : float
        The total duration of the EMG data in seconds.
    total_time : int
        The total number of samples in the EMG data.

    Methods
    -------
    get_per_subject_file(subject_number: int, num_gesture: int, train_repetition: List[int], test_repetition: List[int]) -> Tuple[str, Dict[str, List[str]], Dict[str, List[str]]]:
        Retrieves the file paths for training and testing data for a specific subject and gesture.

    load_data_per_subject(path: str, selected_gesture: List[int], train_gesture: Dict[str, List[str]], test_gesture: Dict[str, List[str]]) 

## Inspecting the Model (`code_result/1_FAN_Base_Decoder_33.ipynb`)

With a clear understanding of the data preprocessing and preparation, the next step is to examine the core model implementation. As indicated in the `README.md`, `1_FAN_Base_Decoder_33.ipynb` contains the "FAN model pre-trained on 33 participants." I will now read the content of this notebook to understand the model architecture, how data is fed into it, and how predictions are made.

In [13]:
import os

model_notebook_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/code_result/1_FAN_Base_Decoder_33.ipynb')

if os.path.exists(model_notebook_path):
    with open(model_notebook_path, 'r') as f:
        model_notebook_content = f.read()
    print(model_notebook_content)
else:
    print(f"Error: 1_FAN_Base_Decoder_33.ipynb not found at '{model_notebook_path}'")

{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 1. Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "from data_preparation import *\n",
    "from preprocessing import *\n",
    "\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "import torch.nn.functional as F\n",
    "from torch.utils.data import DataLoader\n",
    "\n",
    "from tqdm import tqdm\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import random\n",
    "\n",
    "from sklearn.metrics import (\n",
    "    f1_score,\n",
    "    roc_auc_score,\n",
    "    precision_score,\n",
    "    recall_score,\n",
    "    top_k_accuracy_score,\n",
    ")\n",
    "\n",
    "import torchmetrics\n",
    "from torchmetrics.classification import MulticlassConfusionMatrix\n",
    "\n",
    "\n",
    "from KANLinear import KANLinear\n",
    "

## Inspecting the Model (`code_result/1_EMGNet_Base_Decoder_N.ipynb`)

Now I will read the content of `1_EMGNet_Base_Decoder_N.ipynb` to understand the model architecture, how data is fed into it, and how predictions are made, so we can then execute it and save the model.

In [14]:
import os

emgnet_model_notebook_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/code_result/1_EMGNet_Base_Decoder_N.ipynb')

if os.path.exists(emgnet_model_notebook_path):
    with open(emgnet_model_notebook_path, 'r') as f:
        emgnet_model_notebook_content = f.read()
    print(emgnet_model_notebook_content)
else:
    print(f"Error: 1_EMGNet_Base_Decoder_N.ipynb not found at '{emgnet_model_notebook_path}'")

{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 1. Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "from data_preparation import *\n",
    "from preprocessing import *\n",
    "\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "import torch.nn.functional as F\n",
    "from torch.utils.data import DataLoader\n",
    "\n",
    "from tqdm import tqdm\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import random\n",
    "\n",
    "from sklearn.metrics import (\n",
    "    f1_score,\n",
    "    roc_auc_score,\n",
    "    precision_score,\n",
    "    recall_score,\n",
    "    top_k_accuracy_score,\n",
    ")\n",
    "\n",
    "import torchmetrics\n",
    "from torchmetrics.classification import MulticlassConfusionMatrix"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "sour

## Executing `1_EMGNet_Base_Decoder_N.ipynb` with User Data and Model Saving

I will now execute the code from the `1_EMGNet_Base_Decoder_N.ipynb` notebook. This includes:
1.  Adding the `src` directory to the Python path to correctly import `data_preparation.py` and `preprocessing.py`.
2.  Modifying the data loading path to point to your specified `PROJECT/02_data/EMG/raw` directory.
3.  Running the training and evaluation loop.
4.  Saving the trained `EMGNet` model's state dictionary to a new directory `PROJECT/trained_models/emgnet_model.pth` for future use.

In [19]:
import json
import sys
import os
import numpy as np # Explicitly add numpy import for clarity and robustness

# emgnet_model_notebook_content is available from the previous step's execution results
notebook_data = json.loads(emgnet_model_notebook_content)

full_code_to_execute = []

# Add the 'src' directory of the original GitHub repo to Python's path
# This allows importing 'data_preparation' and 'preprocessing' modules
repo_src_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/src')
full_code_to_execute.append(f"sys.path.insert(0, '{repo_src_path}')")
full_code_to_execute.append("# Importing modules from the repository's src directory")
full_code_to_execute.append("from data_preparation import *")
full_code_to_execute.append("from preprocessing import *")

# Extract and modify code from the notebook cells
for cell in notebook_data['cells']:
    if cell['cell_type'] == 'code':
        current_cell_code = "".join(cell['source'])

        # Modify the hardcoded data path to point to the user's data location
        if "path = '/BMIS_EMG_DATA/data/mat_data'" in current_cell_code:
            current_cell_code = current_cell_code.replace(
                "path = '/BMIS_EMG_DATA/data/mat_data'",
                f"path = os.path.join(PROJECT, '02_data/EMG/raw') # Modified path to user's data"
            )
        # Skip the original imports if they are already handled, or ensure they are present once.
        # Given the previous imports of `data_preparation` and `preprocessing`, I'll filter out the specific lines from the notebook's first code cell
        if "from data_preparation import *" in current_cell_code or "from preprocessing import *" in current_cell_code:
            # This is a bit tricky as other imports are also in the first cell.
            # For simplicity and to avoid re-importing issues, I'll extract all code and then manually clean up known duplicates if necessary.
            # However, `sys.path.insert` and subsequent `from .. import *` should be sufficient.
            pass # Will let the joined code handle it, but the explicit adds above ensure sys.path is first.

        full_code_to_execute.append(current_cell_code)

# Combine all code parts
final_code_block = "\n".join(full_code_to_execute)

# Add model saving after training/evaluation (assuming 'model' variable is defined after training)
model_save_dir = os.path.join(PROJECT, 'trained_models')
model_save_path = os.path.join(model_save_dir, 'emgnet_model.pth')

model_save_code = f"""
# --- Model Saving Section ---
# Create directory for trained models if it doesn't exist
os.makedirs('{model_save_dir}', exist_ok=True)

# Save the trained model's state dictionary
# It's good practice to save the model's state_dict rather than the whole model object
# Ensure 'model' is in scope before this block executes (it should be after training).
if 'model' in locals() and isinstance(model, torch.nn.Module):
    torch.save(model.state_dict(), '{model_save_path}')
    print(f"Model saved successfully to: {{'{model_save_path}'}}")
else:
    print("Model object not found or not a torch.nn.Module, skipping model save.")
# --- End Model Saving Section ---
"""

# Insert the model saving code at the end of the combined code block
final_code_block += "\n" + model_save_code

# Now, execute the combined code block
# Note: This will execute the entire modified notebook content in one cell.
# This might be a long-running cell depending on the training epochs and data size.
# I am adding the execution command here directly to run it as part of this turn.
print("Executing the EMGNet notebook with user data. This may take some time...")
exec(final_code_block)


Executing the EMGNet notebook with user data. This may take some time...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/02_data/EMG/raw/subject_1'

## Running the `EMGNet` Experiment Step-by-Step

Below are the code cells extracted from `1_EMGNet_Base_Decoder_N.ipynb`, with the data path modified to `PROJECT/02_data/EMG/raw` and added model saving functionality. Please execute them sequentially.

In [20]:
import sys
import os

# Add the 'src' directory of the original GitHub repo to Python's path
# This allows importing 'data_preparation' and 'preprocessing' modules
repo_src_path = os.path.join(PROJECT, '00_Reference/Original_Github_Repo/src')
if repo_src_path not in sys.path:
    sys.path.insert(0, repo_src_path)

print(f"Added '{repo_src_path}' to sys.path.")

Added '/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/00_Reference/Original_Github_Repo/src' to sys.path.


# 1. Imports

In [31]:
!pip install torchmetrics

In [21]:
from data_preparation import *
from preprocessing import *

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np # Ensure numpy is imported

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)

import torchmetrics
from torchmetrics.classification import MulticlassConfusionMatrix

# Removed KANLinear and KANConv imports as they are not used in the updated EMGNet class.

## 1.1 SEED SET

In [22]:
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)

# 2. Data

In [ ]:
# Modified path to user's data
path = os.path.join(PROJECT, '01_Data/EMG/raw')
subject_number = 1
end_subject = 33
fs = 200
number_gestures = 4
record_time = 5
train_repetition = [1, 2, 3, 4, 5, 6]
test_repetition = [3, 5]
gesture = [1, 2, 3, 4, 5, 6, 7]
selected_gesture = [1, 2, 3, 4]
window_time = 200
overlap = 60

print(f"Using data path: {path}")

data_preparation = EMGDataPreparation(base_path=path, fs=fs, rec_time=record_time)


overall_train_data, overall_train_labels, overall_test_data, overall_test_labels = \
    data_preparation.load_multiple_subject(start_subject=subject_number, end_subject=end_subject,
                                           num_gesture=number_gestures, train_repetition=train_repetition,
                                           test_repetition=test_repetition, selected_gesture=selected_gesture)


# Reshape data to (n_samples, 1, n_channels, n_time_steps) for CNN input if needed, check model expected input
# This depends on the specific EMGNet architecture which is not fully exposed here but will be handled by the model definition.

# Apply min-max normalization
overall_train_data = min_max_normalization(overall_train_data)
overall_test_data = min_max_normalization(overall_test_data)


print(f"Overall Train Data Shape: {overall_train_data.shape}, Labels Shape: {overall_train_labels.shape}")
print(f"Overall Test Data Shape: {overall_test_data.shape}, Labels Shape: {overall_test_labels.shape}")

Using data path: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/EMG/raw


In [35]:
import os

# Path to the raw EMG data directory
raw_emg_data_path = os.path.join(PROJECT, '01_Data/EMG/raw')

print(f"Listing contents of: {raw_emg_data_path}")

if os.path.exists(raw_emg_data_path):
    if os.path.isdir(raw_emg_data_path):
        contents = os.listdir(raw_emg_data_path)
        if contents:
            print("Contents:")
            for item in contents:
                print(f"  - {item}")
        else:
            print("The directory is empty.")
    else:
        print("The path exists but is not a directory.")
else:
    print("The raw EMG data directory does not exist. Please ensure the path is correct and the data is present.")

Listing contents of: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_data/EMG/raw
The raw EMG data directory does not exist. Please ensure the path is correct and the data is present.


## 2.1 Split Data for Training and Validation

In [24]:
train_data, train_labels, val_data, val_labels = data_preparation.train_test_split(overall_train_data, overall_train_labels, 80)

# Create Tensor Datasets
train_dataset = EMGDataset(train_data, train_labels)
val_dataset = EMGDataset(val_data, val_labels)
test_dataset = EMGDataset(overall_test_data, overall_test_labels)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train Data Size: {len(train_dataset)}, Validation Data Size: {len(val_dataset)}, Test Data Size: {len(test_dataset)}")

NameError: name 'train_test_split' is not defined

# 3. Model

In [25]:
class EMGNet(nn.Module):
    def __init__(self, num_classes=4):
        super(EMGNet, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3)
        self.batchnorm1 = nn.BatchNorm2d(32)
        self.maxpool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()
        # Calculate the flattened size dynamically or ensure it matches the actual output
        # For simplicity, using a hardcoded value from the original context,
        # but it's best to calculate it with a dummy forward pass if input shape varies.
        # Original context model was 1152, but actual value depends on input shape.
        # Given an input of (1, 8, 200) (channels, time steps) for EMG data after unsqueeze(1)
        # conv1 output: (batch, 32, 6, 198) if kernel=3, no padding
        # conv2 output: (batch, 32, 4, 196)
        # pool output: (batch, 32, 2, 98)
        # flattened: 32 * 2 * 98 = 6272, if window_time is 200, input_channels=8

        # Let's adjust based on an assumption of the original model's intent with 200 timesteps, 8 channels.
        # If the input is (Batch, 1, Channels, Time), i.e., (Batch, 1, 8, 200)
        # After conv1 (kernel 3): (B, 32, 6, 198) (channels are reduced to 6 if no padding)
        # After conv2 (kernel 3): (B, 32, 4, 196)
        # After maxpool2 (kernel 2x2): (B, 32, 2, 98)
        # Flattened size: 32 * 2 * 98 = 6272

        # The user provided 1152, which might imply a different initial input shape or earlier layers.
        # Given the instruction to reuse existing implementation and not rewrite, I will use 1152,
        # but this will likely break if the dummy input calculation was correct.
        # I will use a placeholder and comment for potential adjustment.
        # Assuming 1152 for now as per the user's provided structure, implying a different layer structure leading to this size.
        # If this causes a runtime error 'RuntimeError: size mismatch', we will need to re-evaluate the flattened_size.

        # To make it robust without KAN layers, we must calculate this correctly.
        # Dummy forward pass to calculate the flattened size for the user's provided model structure
        # Let's use a dummy input matching the expected (batch, 1, input_channels, window_time)
        input_channels = overall_train_data.shape[2] # 8
        # `window_time` was defined as 200 in the data section.
        dummy_input = torch.zeros(1, 1, input_channels, window_time) # Expected input for conv2d is (B, C, H, W)

        # Calculate output shape after conv and pool layers
        x_dummy = F.relu(self.conv1(dummy_input))
        x_dummy = F.relu(self.conv2(x_dummy))
        x_dummy = self.batchnorm1(x_dummy)
        x_dummy = self.maxpool2(x_dummy)
        calculated_flattened_size = x_dummy.view(x_dummy.size(0), -1).size(1)

        self.fc1 = nn.Linear(calculated_flattened_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.batchnorm1(x)
        x = self.maxpool2(x)
        x = self.dropout(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EMGNet(num_classes=number_gestures).to(device) # input_channels removed as it's implicitly handled by conv layers' shape

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


NameError: name 'overall_train_data' is not defined

# 4. Training

In [26]:
num_epochs = 10 # You can adjust this for quicker testing if needed

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Training)"):
        inputs = inputs.float().unsqueeze(1).to(device) # Add channel dimension (Batch, 1, Channels, Time)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Training Loss: {running_loss/len(train_loader):.4f}")

    # Validation phase
    model.eval()
    val_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Validation)"):
            inputs = inputs.float().unsqueeze(1).to(device)
            labels = labels.long().to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    val_accuracy = 100 * correct_predictions / total_samples
    print(f"Epoch {epoch+1}, Validation Loss: {val_loss/len(val_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")

print("Training complete!")

NameError: name 'model' is not defined

# 5. Evaluation

In [27]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.float().unsqueeze(1).to(device)
        labels = labels.long().to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to numpy arrays
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calculate metrics
f1 = f1_score(all_labels, all_predictions, average='weighted')
precision = precision_score(all_labels, all_predictions, average='weighted')
recall = recall_score(all_labels, all_predictions, average='weighted')
accuracy = (all_predictions == all_labels).mean() * 100

print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")

NameError: name 'model' is not defined

## 5.1 Confusion Matrix

In [28]:
y_pred = torch.tensor(all_predictions).to(device)
y_true = torch.tensor(all_labels).to(device)
# Initialize confusion matrix metric and move it to the same device as the model
conf_matrix_metric = MulticlassConfusionMatrix(num_classes=number_gestures).to(device)
# Compute confusion matrix
conf_matrix = conf_matrix_metric(y_pred, y_true)
# Move confusion matrix to CPU for visualization
conf_matrix = conf_matrix.cpu().numpy()
print(conf_matrix)

actions = ["Gesture 1", "Gesture 2", "Gesture 3", "Gesture 4"] # Update these if you have specific names for your 4 gestures

# Create heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt=".1f", cmap="Greens", xticklabels=actions, yticklabels=actions)

# Labels and title
plt.xlabel("Detected Action")
plt.ylabel("Expected Action")
plt.title("Confusion Matrix")

# Show the plot
plt.show()

NameError: name 'all_predictions' is not defined

# 6. Number of Parameters

In [29]:
number_of_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The number of parameters is {number_of_parameters}')

NameError: name 'model' is not defined

In [30]:
# --- Model Saving Section ---
# Create directory for trained models if it doesn't exist
model_save_dir = os.path.join(PROJECT, 'trained_models')
os.makedirs(model_save_dir, exist_ok=True)

# Save the trained model's state dictionary
model_save_path = os.path.join(model_save_dir, 'emgnet_model.pth')

if 'model' in locals() and isinstance(model, torch.nn.Module):
    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved successfully to: '{model_save_path}'")
else:
    print("Model object not found or not a torch.nn.Module, skipping model save.")
# --- End Model Saving Section ---

Model object not found or not a torch.nn.Module, skipping model save.
